<a href="https://colab.research.google.com/github/sujitx-vs/Deep-Neural-Networks---PGCP_AI/blob/main/DNN_Day4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers,callbacks
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix


In [ ]:
iris = load_iris()
X, y = iris.data, iris.target

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,stratify=y)
X_train.shape


In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [ ]:
print("Train:", X_train.shape, "Test:",X_test.shape)

In [ ]:
print(y_train)

One hot encode labels

In [ ]:
y_train_oh = keras.utils.to_categorical(y_train, num_classes=3)
y_test_oh  = keras.utils.to_categorical(y_test,  num_classes=3)

In [ ]:
print(y_train_oh)

In [ ]:
def build_model(input_dim=4, num_classes=3):
    model = keras.Sequential([
        # Input layer
        layers.Input(shape=(input_dim,)),

        # Hidden layer 1 — 16 neurons, ReLU activation
        layers.Dense(16, activation='relu'),
        layers.BatchNormalization(),   # stabilise activations across batches
        layers.Dropout(0.2),           # randomly zero 20% of neurons → reduces overfitting

        # Hidden layer 2 — 8 neurons, ReLU activation
        layers.Dense(8, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.2),

        # Output layer — 3 neurons (one per class), Softmax → probability distribution
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

model = build_model()
model.summary()

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
early_stop = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=20,          # stop if val_loss doesn't improve for 20 epochs
    restore_best_weights=True  # revert to the epoch with best val_loss
)
#not required for small datasets
#lr_scheduler = callbacks.ReduceLROnPlateau(
 #   monitor='val_loss',
  #  factor=0.5,           # halve the LR when plateauing
   # patience=10,
    #min_lr=1e-6,
    #verbose=1
#)

history = model.fit(
    X_train, y_train_oh,
    epochs=100,
    batch_size=16,
    validation_split=0.15,  # use 15% of training data for validation
    callbacks=[early_stop] #, lr_scheduler],

)

print(f"\nTraining stopped at epoch: {early_stop.stopped_epoch}")

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test_oh, verbose=0)
print(f"Test loss:     {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f} ({test_acc*100:.1f}%)")

# Detailed per-class report
y_pred_prob = model.predict(X_test)          # shape (30, 3) — probability per class
y_pred      = np.argmax(y_pred_prob, axis=1) # take highest-probability class

print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=iris.target_names,
            yticklabels=iris.target_names)
plt.title('Confusion matrix — Neural network')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()


In [ ]:
new_sample = np.array([[5.1, 3.5, 1.4, 0.2]])

# Scaling
new_scaled = scaler.transform(new_sample)

# Predict
probs = model.predict(new_scaled)[0]
probs = np.round(probs, 3)
print(f"Probabilities: {probs}")
pred_class = np.argmax(probs)
pred_name  = iris.target_names[pred_class]
print(f"Predicted class: {pred_name}")
